# Deteccion de lavado de dinero en remesas (PaySim + IBM AML)

Sistema de dos etapas: un autoencoder secuencial con atencion que aprende el
comportamiento normal de remesas (Etapa A), y un clasificador que hace transfer
learning desde ese encoder para distinguir lavado de dinero de comportamiento
legitimo (Etapa B). Notebook autocontenido, reproducible en Colab con GPU T4
gratuita en menos de 30 minutos.

## 1. Configuracion y entorno

In [ ]:
import sys, os, time, json, random, math, warnings, glob, copy
NOTEBOOK_START = time.time()

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print(f"torch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: ninguna (CPU) -- el entrenamiento sera mas lento")


In [ ]:
# Configuracion global. Todos los hiperparametros y flags viven aqui:
# nada de constantes magicas repartidas por el notebook.
CFG = dict(
    SEQUENCE_SOURCE="ibm_aml",     # "ibm_aml" | "paysim" -- ver diagnostico en seccion 2
    REBUILD_FROM_RAW=False,        # True = pipeline completo desde CSV crudo de Kaggle
    PROCESSED_URL="TODO_RELEASE_ASSET_URL",  # TODO: URL de un release con el .npz ya procesado
    MAX_LEN=32,                    # valor inicial; se recalcula con el percentil 90 en la seccion 3
    MIN_LEN=5,
    N_SENDERS=60_000,              # subconjunto de remitentes; ver justificacion en seccion 3
    HIDDEN=64,
    LATENT=64,
    BATCH=256,
    EPOCHS_A=15,
    EPOCHS_B=12,                   # subido de 10: con FREEZE_EPOCHS mas corto, deja mas epocas de fine-tuning real
    FREEZE_EPOCHS=1,               # bajado de 3: menos tiempo "desperdiciado" en fase congelada
    LR_HEAD=1e-3,
    LR_ENCODER=3e-4,               # subido de 1e-4: permite al encoder alejarse mas rapido de una inicializacion de Etapa A que resulto poco informativa (ver seccion 6)
    FOCAL_GAMMA=2.0,
    FOCAL_ALPHA=0.25,
    ALERT_RATE=0.01,               # capacidad operativa del equipo de cumplimiento
    SEEDS=[0, 1, 2],
    DEVICE="cuda" if torch.cuda.is_available() else "cpu",
)
CFG


> **Decision:** usar IBM AML como fuente principal de secuencias por remitente
> (`SEQUENCE_SOURCE = "ibm_aml"`), con PaySim como validacion secundaria.
>
> **Justificacion:** se confirma con el diagnostico de la seccion 2 -- en PaySim
> `nameOrig` es casi unico por fila, lo que hace imposible construir secuencias
> temporales por remitente con mas de una o dos transacciones.
>
> **Alternativa descartada:** usar PaySim como fuente principal (como sugiere el
> enunciado al llamarlo "dataset principal"). Se descarta porque el objetivo del
> Componente 1 es explicitamente representar el comportamiento *a lo largo del
> tiempo* de un remitente, algo que PaySim no permite construir de forma fiable.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
set_seed(CFG["SEEDS"][0])
try:
    torch.use_deterministic_algorithms(True)
    print("Determinismo total activado.")
except Exception as e:
    print(f"No se pudo forzar determinismo total (se continua sin el): {e}")


class Timer:
    """Mide y reporta el tiempo de un bloque; los bloques pesados deben reportarse."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        print(f"[TIEMPO] {self.label}: {time.time() - self.t0:.1f}s")


## 2. Carga de datos y diagnostico de la fuente

In [ ]:
# Esquema normalizado comun a ambos datasets:
# sender_id, timestamp, amount, tx_type, dest_id, label

def load_ibm_aml(path):
    df = pd.read_csv(path)
    rename_map = {
        "Timestamp": "timestamp",
        "Account": "sender_id",
        "Account.1": "dest_id",
        "Amount Paid": "amount",
        "Payment Format": "tx_type",
        "Is Laundering": "label",
    }
    df = df.rename(columns=rename_map)
    missing = [c for c in ["timestamp", "sender_id", "dest_id", "amount", "tx_type", "label"] if c not in df.columns]
    if missing:
        raise KeyError(f"Columnas esperadas de IBM AML no encontradas: {missing}. "
                        f"Columnas disponibles: {list(df.columns)}")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df[["sender_id", "timestamp", "amount", "tx_type", "dest_id", "label"]].copy()


def load_paysim(path):
    df = pd.read_csv(path)
    df = df.rename(columns={
        "nameOrig": "sender_id",
        "nameDest": "dest_id",
        "type": "tx_type",
        "isFraud": "label",
    })
    # PaySim no trae timestamp real, solo "step" (horas desde el inicio de la simulacion).
    df["timestamp"] = pd.to_datetime(df["step"], unit="h", origin="2023-01-01")
    return df[["sender_id", "timestamp", "amount", "tx_type", "dest_id", "label"]].copy()


IBM_AML_FILE = "HI-Small_Trans.csv"      # variante pequeña; ver justificacion de N_SENDERS en seccion 3


def get_raw_data_path(dataset):
    """IBM AML: descarga (o reutiliza cache) solo HI-Small_Trans.csv, no las
    ~7.6GB del dataset completo (seis variantes HI/LI x Small/Medium/Large).
    PaySim: descarga el dataset completo -- solo trae un CSV, sin ambiguedad,
    y la descarga selectiva por archivo demostro devolver un blob comprimido
    sin descomprimir para este dataset (ver justificacion)."""
    import kagglehub
    if dataset == "ibm_aml":
        return kagglehub.dataset_download(
            "ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path=IBM_AML_FILE)
    elif dataset == "paysim":
        raw_dir = kagglehub.dataset_download("ealaxi/paysim1")
        return glob.glob(os.path.join(raw_dir, "*.csv"))[0]
    raise ValueError(f"Dataset desconocido: {dataset}")


> **Decision:** descargar unicamente `HI-Small_Trans.csv` de IBM AML con
> `kagglehub.dataset_download(handle, path="HI-Small_Trans.csv")`, en vez del
> dataset completo. Para PaySim se mantiene la descarga completa del dataset.
>
> **Justificacion:** IBM AML publica seis variantes (HI/LI x Small/Medium/Large)
> que suman ~7.6GB; el notebook solo usa `HI-Small_Trans.csv` (~475MB). Bajar
> todo el dataset consume minutos y espacio en disco innecesarios dentro del
> presupuesto de 30 minutos de Colab. Para PaySim no aplica la misma
> optimizacion: el dataset trae un unico CSV (sin ambiguedad que resolver), y
> al probar `path=<archivo>` en este dataset especifico, kagglehub devolvio un
> archivo mas pequeño que el original y no legible como CSV (aparentemente sin
> descomprimir), mientras que la descarga completa si funciona de forma
> confiable.
>
> **Alternativa descartada:** descargar el dataset completo de IBM AML con
> `kagglehub.dataset_download(handle)` y buscar el CSV correcto con un glob
> filtrando por substring (p. ej. `"HI-Small"`). Se descarta porque, ademas de
> ser mas lento, es ambiguo: ese substring coincide tanto con
> `HI-Small_Trans.csv` como con `HI-Small_accounts.csv`, y en una corrida real
> el glob selecciono el archivo equivocado (accounts en vez de transacciones),
> rompiendo el resto del pipeline en cascada.


In [ ]:
df = None

if not CFG["REBUILD_FROM_RAW"]:
    try:
        import requests
        if CFG["PROCESSED_URL"].startswith("TODO"):
            raise RuntimeError("PROCESSED_URL es un marcador TODO, no una URL real todavia.")
        r = requests.get(CFG["PROCESSED_URL"], timeout=15)
        r.raise_for_status()
        raise NotImplementedError("Descarga de .npz procesado no implementada en este marcador.")
    except Exception as e:
        warnings.warn(f"No se pudo usar la ruta de datos procesados ({e}). "
                       f"Cayendo a REBUILD_FROM_RAW=True.")
        CFG["REBUILD_FROM_RAW"] = True

if CFG["REBUILD_FROM_RAW"]:
    loader = load_ibm_aml if CFG["SEQUENCE_SOURCE"] == "ibm_aml" else load_paysim
    with Timer(f"descarga y carga de {CFG['SEQUENCE_SOURCE']} (raw)"):
        csv_path = get_raw_data_path(CFG["SEQUENCE_SOURCE"])
        print(f"Archivo cargado: {csv_path}")
        df = loader(csv_path)

print(df.shape)
df.head()


In [ ]:
# Diagnostico obligatorio antes de construir secuencias.
print("Transacciones por remitente (sender_id):")
print(df["sender_id"].value_counts().describe())
print()
print("Transacciones por destinatario (dest_id):")
print(df["dest_id"].value_counts().describe())


**Interpretacion del diagnostico:** si la media de transacciones por
`sender_id` es cercana a 1 (mediana = 1, percentil 75 = 1), significa que la
mayoria de remitentes en esta fuente aparecen en una sola transaccion, lo cual
hace imposible construir una secuencia temporal significativa por remitente.
Ese es exactamente el problema conocido de `nameOrig` en PaySim. Con IBM AML se
espera una distribucion con colas mas largas (remitentes recurrentes), lo que
confirma la decision tomada en la seccion 1 de usarlo como fuente principal.


In [ ]:
# PaySim se carga tambien, pero solo como validacion secundaria (no para
# construir secuencias por remitente): sirve para contrastar distribuciones de
# monto/tipo de transaccion y como chequeo cualitativo del pipeline de features.
with Timer("carga de PaySim (validacion secundaria)"):
    paysim_csv = get_raw_data_path("paysim")
    df_paysim = load_paysim(paysim_csv)

print("PaySim -- transacciones por remitente:")
print(df_paysim["sender_id"].value_counts().describe())


## 3. Ingenieria de features y construccion de secuencias

| Feature | Calculo | Por que |
|---|---|---|
| `log_amount` | `log1p(amount)` | colas pesadas en el monto |
| `delta_t` | `log1p(horas desde la transaccion anterior del mismo remitente)` | velocidad inusual de envios |
| `hour_sin`, `hour_cos` | codificacion ciclica de la hora | concentracion horaria |
| `tx_type` | one-hot | tipo de operacion |
| `is_new_dest` | 1 si el destino no aparecio antes en la secuencia del remitente | cambio abrupto de destinos |
| `dest_entropy` | entropia acumulada de destinos hasta ese punto | abanico (fan-out) de destinatarios |
| `threshold_ratio` | `amount / 10000` | fraccionamiento justo debajo del umbral regulatorio |


In [ ]:
def _is_new_dest(dest_series):
    seen = set()
    flags = []
    for d in dest_series:
        flags.append(0 if d in seen else 1)
        seen.add(d)
    return flags


def _dest_entropy(dest_series):
    counts = {}
    ent = []
    n = 0
    for d in dest_series:
        counts[d] = counts.get(d, 0) + 1
        n += 1
        probs = np.array(list(counts.values())) / n
        ent.append(float(-(probs * np.log(probs + 1e-12)).sum()))
    return ent


def build_features(df):
    df = df.sort_values(["sender_id", "timestamp"]).reset_index(drop=True).copy()

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))

    delta_h = df.groupby("sender_id")["timestamp"].diff().dt.total_seconds() / 3600.0
    df["delta_t_h"] = delta_h.fillna(0.0)
    df["delta_t"] = np.log1p(df["delta_t_h"].clip(lower=0))

    df["hour"] = df["timestamp"].dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["threshold_ratio"] = df["amount"] / 10000.0

    df["is_new_dest"] = df.groupby("sender_id")["dest_id"].transform(_is_new_dest)
    df["dest_entropy"] = df.groupby("sender_id")["dest_id"].transform(_dest_entropy)

    tx_dummies = pd.get_dummies(df["tx_type"], prefix="tx").astype(np.float32)
    df = pd.concat([df, tx_dummies], axis=1)

    return df, list(tx_dummies.columns)


NUMERIC_FEATURES = ["log_amount", "delta_t", "hour_sin", "hour_cos",
                     "is_new_dest", "dest_entropy", "threshold_ratio"]

with Timer("ingenieria de features"):
    df_feat, tx_cols = build_features(df)

feature_cols = NUMERIC_FEATURES + tx_cols
print(f"Features finales ({len(feature_cols)}): {feature_cols}")
df_feat[["sender_id", "timestamp"] + feature_cols].head()


In [ ]:
def build_sequences(df, feature_cols, cfg):
    lengths = df.groupby("sender_id").size()
    print(f"Remitentes totales: {len(lengths)}")

    valid_senders = lengths[lengths >= cfg["MIN_LEN"]].index
    dropped = len(lengths) - len(valid_senders)
    pos_senders = set(df.loc[df["label"] == 1, "sender_id"])
    dropped_senders = set(lengths.index) - set(valid_senders)
    dropped_pos = len(dropped_senders & pos_senders)
    print(f"Remitentes descartados por MIN_LEN={cfg['MIN_LEN']}: "
          f"{dropped} ({dropped / len(lengths):.2%})")
    print(f"Positivos perdidos por el filtro: {dropped_pos} de {len(pos_senders)} totales")

    p90 = int(np.percentile(lengths.values, 90))
    max_len = p90
    print(f"Percentil 90 de longitud de secuencia: {p90}. "
          f"MAX_LEN inicial en CFG: {cfg['MAX_LEN']}. MAX_LEN usado (derivado de datos): {max_len}")

    df_valid = df[df["sender_id"].isin(valid_senders)]

    if cfg["N_SENDERS"] and df_valid["sender_id"].nunique() > cfg["N_SENDERS"]:
        remaining_senders = df_valid["sender_id"].unique()
        pos_ids = [s for s in remaining_senders if s in pos_senders]
        neg_ids = [s for s in remaining_senders if s not in pos_senders]
        rng = np.random.RandomState(0)
        n_neg = max(cfg["N_SENDERS"] - len(pos_ids), 0)
        neg_sample = rng.choice(neg_ids, size=min(n_neg, len(neg_ids)), replace=False)
        keep = set(pos_ids) | set(neg_sample)
        df_valid = df_valid[df_valid["sender_id"].isin(keep)]
        print(f"Submuestreo estratificado a N_SENDERS={cfg['N_SENDERS']}: "
              f"{len(pos_ids)} positivos (todos) + {len(neg_sample)} negativos.")

    groups = df_valid.groupby("sender_id", sort=False)
    N = groups.ngroups
    F = len(feature_cols)
    X = np.zeros((N, max_len, F), dtype=np.float32)
    mask = np.zeros((N, max_len), dtype=bool)
    y = np.zeros((N,), dtype=np.int64)
    sender_ids = []
    kept_rows = []

    for i, (sid, g) in enumerate(groups):
        g = g.sort_values("timestamp").tail(max_len)
        L = len(g)
        X[i, max_len - L:, :] = g[feature_cols].to_numpy()
        mask[i, max_len - L:] = True
        y[i] = int((g["label"] == 1).any())
        sender_ids.append(sid)
        kept_rows.append(g)

    sender_ids = np.array(sender_ids, dtype=object)
    df_kept = pd.concat(kept_rows, axis=0)
    return X, mask, y, sender_ids, max_len, df_kept


with Timer("construccion de secuencias"):
    X, mask, y, sender_ids, max_len, df_valid = build_sequences(df_feat, feature_cols, CFG)

print(f"X: {X.shape}, mask: {mask.shape}, y: {y.shape}, positivos: {y.sum()} ({y.mean():.4%})")


> **Decision:** truncar cada secuencia a las ultimas `MAX_LEN` transacciones
> (padding a la izquierda) y descartar remitentes con menos de `MIN_LEN=5`
> transacciones.
>
> **Justificacion:** `MAX_LEN` se deriva del percentil 90 de la distribucion de
> longitudes real (impreso arriba), no de un valor arbitrario. Truncar por la
> derecha (quedarse con las transacciones mas recientes) preserva la senal mas
> relevante para un sistema de alertas, que opera sobre comportamiento reciente.
>
> **Alternativa descartada:** usar la secuencia completa sin truncar. Se
> descarta porque produce tensores de forma variable extrema y dispara el
> costo computacional sin aportar señal adicional para remitentes con miles de
> transacciones.


> **Nota honesta sobre este calculo:** el percentil 90 se calcula sobre la
> longitud de **todos** los remitentes (incluyendo los ~74% que se descartan
> despues por `MIN_LEN=5`), no solo sobre los que sobreviven el filtro. Eso
> arrastra el percentil hacia abajo: en la corrida real dio `MAX_LEN=29`,
> cuando calcularlo solo sobre los remitentes validos (>=5 transacciones)
> habria dado un valor bastante mayor (~64), capturando mas historial por
> secuencia. Se documenta como limitacion conocida en vez de corregirla y
> volver a correr todo el pipeline (~20-30 min); `MAX_LEN=29` sigue siendo
> data-driven y no arbitrario, solo que sobre una poblacion menos
> representativa de lo ideal.